# Pipeline EDA — Opción B: comercio electrónico

Pipeline de 10 etapas sobre *eCommerce behavior data (multi-category store)* + componente sintético.
La lógica vive en `pipeline_eda/`; este notebook solo importa, ejecuta y muestra resultados.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Busca hacia arriba la carpeta que contiene pipeline_eda/, sin depender del cwd del kernel.
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pipeline_eda").is_dir())
sys.path.insert(0, str(RAIZ))

In [ ]:
from pipeline_eda.etapa0_ingesta import TAMANOS, leer_csv
from pipeline_eda.orquestador import Config, Pipeline

In [ ]:
from pipeline_eda.etapa0_ingesta import leer_csv

eventos = leer_csv(RAIZ / "data" / "2019-Oct-10M.csv")

In [ ]:
next(eventos)

Event(event_time='2019-10-01 00:00:00 UTC', event_type='view', product_id=44600062, category_id=2103807459595387724, category_code='', brand='shiseido', price=35.79, user_id=541312140, user_session='72d76fde-8bb3-4e00-8c23-a032dfed738c', order_id=None, tipo_envio=None, card_id=None, device_id=None, is_fraud=False, is_dup=False)

## 1. Datos

In [ ]:
TAMANO = "100K"  # uno de TAMANOS: 1K, 10K, 100K, 1M, 10M
# Eventos reales; la etapa 0 agrega el componente sintético (con SEMILLA).
fuente = leer_csv(RAIZ / "data" / "2019-Oct-10M.csv", limite=TAMANOS[TAMANO])

## 2. Ejecución de punta a punta (etapas 0 → 9)

In [ ]:
config = Config(seed=42)
pipe = Pipeline(config)
pipe.run(fuente)

## 3. Métricas por etapa

In [ ]:
pipe.metricas()

## 4. Prioridades y fraude (etapas 2 y 6)

In [ ]:
pipe.e9.alertas[:10]

## 5. Consultas de productos (etapas 3 y 4)

In [ ]:
pipe.e3.obtener_producto(1004237)
pipe.e4.productos_en_rango_precio(100.0, 200.0)[:10]

## 6. Productos relacionados (etapas 5 y 7)

In [ ]:
pid = 1004237
pipe.e5.similares(pid, k=10), pipe.e7.relacionados(pid, k=10), pipe.e7.similares_vectoriales(pid, k=10)

## 7. Grafo de co-compra (etapa 8)

In [ ]:
pr = pipe.e8.pagerank()
comunidades = pipe.e8.comunidades()
sorted(pr.items(), key=lambda x: -x[1])[:10], pipe.e8.modularidad(comunidades)

## 8. Servicio (etapa 9)

In [ ]:
pipe.e9.autocompletar("electronics.smart", k=5)

In [ ]:
pipe.e9.populares(k=10), pipe.e9.recomendar(pid, k=10)

In [ ]:
pipe.e9.dashboard(pipe.metricas())